In [ ]:
"""
Notes from Shuyang Yao, 2026-03-18:

To run this script, download the repository mylib from
https://github.com/YAO-Shuyang/MazeProject
and rename the folder to "mylib", then place it in your corresponding conda environments.
"""

from mylib.statistic_test import *


def _file_between(b1: int, b2: int):
    path = DFS(b1, b2, maze_type=1, nx=12)
    return path[1:-1]

def behavioral_info_per_mouse(trace, s: int):
    mouse = trace['MiceID']
    beg, end = LapSplit(trace, "DSPMaze")
    rts = classify_lap(spike_nodes_transform(trace['correct_nodes'], 12), beg, 1)
    
    behav_nodes = spike_nodes_transform(trace['correct_nodes'], 12)
    behav_time = trace['correct_time']/1000
    behav_pos = trace['correct_pos']/10
    assert len(behav_nodes) == len(behav_time) == len(behav_pos)
    
    processed_nodes = []
    processed_speed = []
    processed_lap = []
    processed_rt = []
    
    for i in tqdm(range(len(beg))):
        is_discarded_lap = False
        lap_nodes = []
        lap_speed = []
        lap_lap = []
        lap_rt = []
        
        is_displacement = np.where(np.diff(behav_nodes[beg[i]:end[i]+1]) != 0)[0]
        dx = np.diff(behav_pos[beg[i]:end[i]+1, 0])
        dy = np.diff(behav_pos[beg[i]:end[i]+1, 1])
        dl = np.sqrt(dx**2 + dy**2)
        dl = np.append(dl, 0)
        dt = np.diff(behav_time[beg[i]:end[i]+1])
        dt = np.append(dt, dt[-1])
        
        beg_dis = np.concatenate(([0], is_displacement + 1))
        end_dis = np.concatenate((is_displacement+1, [end[i] - beg[i]]))
        
        for j in range(len(beg_dis)):
            lap_nodes.append(int(behav_nodes[beg[i]+beg_dis[j]]))
            speed = (np.sum(dl[beg_dis[j]:end_dis[j]])) / (np.sum(dt[beg_dis[j]:end_dis[j]]) + 1e-6)
            lap_speed.append(speed)
            lap_lap.append(i)
            lap_rt.append(rts[i])
        
        if lap_nodes[-1] != 144:
            lap_nodes.append(144)
            lap_speed.append(0)
            lap_lap.append(i)
            lap_rt.append(rts[i]) 
        
        if lap_nodes[0] != CP_DSPs[1][rts[i]][0]:
            lap_nodes.insert(0, CP_DSPs[1][rts[i]][0])
            lap_speed.insert(0, 0)
            lap_lap.insert(0, i)
            lap_rt.insert(0, rts[i])
        
        # Check if the nodes are spatially connected
        if len(lap_nodes) > 1:
            finished_k = 0
            while finished_k < len(lap_nodes) - 1:
                G = cp.copy(maze1_graph)
                for k in range(finished_k, len(lap_nodes)-1):
                    if lap_nodes[k+1] not in G[lap_nodes[k]]:
                        inserted_nodes = _file_between(lap_nodes[k], lap_nodes[k+1])
                        n_inserted = len(inserted_nodes)

                        if n_inserted > 2:
                            # 5 Exceptional laps with more than 2 inserted nodes, which are likely due to tracking errors. 
                            # We will discard these laps.
                            if mouse == 10212 and s == 2 and i == 30:
                                pass
                            elif mouse == 10224 and s == 1 and i == 58:
                                is_discarded_lap = True
                            elif mouse == 10224 and s == 4 and i == 85:
                                is_discarded_lap = True
                            elif mouse == 10232 and s == 3 and i == 60:
                                is_discarded_lap = True
                            elif mouse == 10232 and s == 3 and i == 98:
                                lap_nodes = lap_nodes[182:]
                                lap_speed = lap_speed[182:]
                                lap_lap = lap_lap[182:]
                                lap_rt = lap_rt[182:]
                                finished_k = 0
                                break
                            else:
                                print(lap_nodes[k-10:k+10])
                                
                                plt.figure(figsize=(4, 4))
                                ax = plt.gca()
                                ax.plot(behav_pos[beg[i]:end[i]+1, 0]/2-0.5, behav_pos[beg[i]:end[i]+1, 1]/2-0.5, marker='o', markersize=3, markeredgewidth=0, color='k')
                                DrawMazeProfile(axes=ax, color='k')
                                ax.invert_yaxis()
                                plt.show()
                                raise ValueError(
                                    f"{mouse}, S{s}, lap {i}:"
                                    f"  k = {k}\n"
                                    f"  {n_inserted} nodes, ie {inserted_nodes}, inserted between {lap_nodes[k]} and {lap_nodes[k+1]}.\n"
                                    f"  Neighbors of {lap_nodes[k]}: {G[lap_nodes[k]]}\n"
                                    f"  Neighbors of {lap_nodes[k+1]}: {G[lap_nodes[k+1]]}\n"
                                )
                        finished_k = k + n_inserted + 1
                        
                        lap_nodes = lap_nodes[:k+1] + inserted_nodes + lap_nodes[k+1:]
                        lap_speed = lap_speed[:k+1] + np.linspace(lap_speed[k], lap_speed[k+1], n_inserted+2)[1:-1].tolist() + lap_speed[k+1:]
                        lap_lap = lap_lap[:k+1] + [lap_lap[k]]*n_inserted + lap_lap[k+1:]
                        lap_rt = lap_rt[:k+1] + [lap_rt[k]]*n_inserted + lap_rt[k+1:]
                        break
                    
                    finished_k = k + 1
        
        if is_discarded_lap:
            continue
        
        # Double Check neighbor connectivity:
        for k in range(len(lap_nodes)-1):
            if lap_nodes[k+1] not in G[lap_nodes[k]]:
                raise ValueError(
                    f"After insertion, nodes {lap_nodes[k]} and {lap_nodes[k+1]} are still not spatially connected in lap {i}.\n"
                    f"Neighbors of {lap_nodes[k]}: {G[lap_nodes[k]]}\n"
                    f"Neighbors of {lap_nodes[k+1]}: {G[lap_nodes[k+1]]}\n"
                )
        if lap_nodes[-1] != 144:
            raise ValueError(f"Lap {i} does not end with node 144 after processing.")
        if lap_nodes[0] != CP_DSPs[1][rts[i]][0]:
            raise ValueError(f"Lap {i} does not start with the correct node after processing.")
        # Assure no consecutive bins are the same.
        assert np.where(np.diff(lap_nodes) == 0)[0].size == 0, f"Lap {i} has consecutive nodes that are the same after processing."
                    
        processed_lap.extend(lap_lap)
        processed_nodes.extend(lap_nodes)
        processed_speed.extend(lap_speed)
        processed_rt.extend(lap_rt)
        
    return (
        np.array(processed_lap, np.int64), 
        np.array(processed_nodes, np.int64), 
        np.array(processed_speed, np.float64), 
        np.array(processed_rt, np.float64)
    )

save_dir = r"D:\Softwares\Anaconda2025\envs\models\Lib\site-packages\rtrv_models\data"
for mouse in [10212, 10224, 10227, 10232]:
    print(f"Processing mouse {mouse}...")
    process = {
        "MouseID": [],
        "Session": [],
        "Nodes": [],
        "Speed": [],
        "Route": [],
        "Lap": []
    }
    idx = np.where((f2['MiceID'] == mouse))[0]
    for s, file in enumerate(f2['Trace File'][idx]):
        with open(file, 'rb') as f:
            trace = pickle.load(f)
            res = behavioral_info_per_mouse(trace, s)
            process['MouseID'].append([mouse]*len(res[0]))
            process['Session'].append([s]*len(res[0]))
            process['Nodes'].append(res[1].tolist())
            process['Speed'].append(res[2].tolist())
            process['Route'].append(res[3].tolist())
            process['Lap'].append(res[0].tolist())
        
    for key in process:
        process[key] = np.concatenate(process[key])
    
    with open(os.path.join(save_dir, f"DSP_{mouse}.pkl"), 'wb') as f:
        pickle.dump(process, f)


Processing mouse 10212...


100%|██████████| 100/100 [00:00<00:00, 1369.86it/s]


Processing mouse 10224...


100%|██████████| 101/101 [00:00<00:00, 1942.31it/s]


Processing mouse 10227...


100%|██████████| 95/95 [00:00<00:00, 2262.12it/s]


Processing mouse 10232...


100%|██████████| 130/130 [00:00<00:00, 2063.25it/s]


In [1]:
from mylib.statistic_test import *

def _file_between(b1: int, b2: int, mz: int = 1):
    path = DFS(b1, b2, maze_type=mz, nx=12)
    return path[1:-1]

def behavioral_info_per_mouse_pretrain(trace, s: int):
    mouse = trace['MiceID']
    beg, end = LapSplit(trace, "CrossMaze")
    rts = classify_lap(spike_nodes_transform(trace['correct_nodes'], 12), beg, 1)
    
    behav_nodes = spike_nodes_transform(trace['correct_nodes'], 12)
    behav_time = trace['correct_time']/1000
    behav_pos = trace['correct_pos']/10
    assert len(behav_nodes) == len(behav_time) == len(behav_pos)
    
    processed_nodes = []
    processed_speed = []
    processed_lap = []
    
    for i in tqdm(range(len(beg))):
        
        is_discarded_lap = False
        lap_nodes = []
        lap_speed = []
        lap_lap = []
        lap_t = []
        
        is_displacement = np.where(np.diff(behav_nodes[beg[i]:end[i]+1]) != 0)[0]
        dx = np.diff(behav_pos[beg[i]:end[i]+1, 0])
        dy = np.diff(behav_pos[beg[i]:end[i]+1, 1])
        dl = np.sqrt(dx**2 + dy**2)
        dl = np.append(dl, 0)
        dt = np.diff(behav_time[beg[i]:end[i]+1])
        dt = np.append(dt, dt[-1])
        
        beg_dis = np.concatenate(([0], is_displacement + 1))
        end_dis = np.concatenate((is_displacement+1, [end[i] - beg[i]]))
        
        for j in range(len(beg_dis)):
            lap_nodes.append(int(behav_nodes[beg[i]+beg_dis[j]]))
            speed = (np.sum(dl[beg_dis[j]:end_dis[j]])) / (np.sum(dt[beg_dis[j]:end_dis[j]]) + 1e-6)
            lap_speed.append(speed)
            lap_lap.append(i)
            lap_t.append(np.mean(behav_time[beg[i]+beg_dis[j]:beg[i]+end_dis[j]]))
        
        if lap_nodes[-1] != 144:
            lap_nodes.append(144)
            lap_speed.append(0)
            lap_lap.append(i)
            lap_t.append(behav_time[end[i]])
        
        if lap_nodes[0] != 1:
            lap_nodes.insert(0, 1)
            lap_speed.insert(0, 0)
            lap_lap.insert(0, i)
            lap_t.insert(0, behav_time[beg[i]])
        
        # Check if the nodes are spatially connected
        if len(lap_nodes) > 1:
            finished_k = 0
            while finished_k < len(lap_nodes) - 1:
                G = cp.copy(maze1_graph) if trace['maze_type'] == 1 else cp.copy(maze2_graph)
                for k in range(finished_k, len(lap_nodes)-1):
                    if lap_nodes[k+1] not in G[lap_nodes[k]]:
                        inserted_nodes = _file_between(lap_nodes[k], lap_nodes[k+1], mz=trace['maze_type'])
                        n_inserted = len(inserted_nodes)

                        if n_inserted > 2:
                            if mouse == 10232 and s == 0 and i == 2:
                                lap_nodes = lap_nodes[696:]
                                lap_speed = lap_speed[696:]
                                lap_lap = lap_lap[696:]
                                lap_t = lap_t[696:]
                                finished_k = 0
                                break
                            else:
                                print(lap_nodes[k-10:k+10])
                                
                                D = GetDMatrices(trace['maze_type'], 12)
                                dis = D[np.asarray(lap_nodes)-1, 0]
                                
                                plt.Figure(figsize=(4, 4))
                                ax = plt.gca()
                                ax.plot(dis, lap_t)
                                plt.show()
                                
                                plt.figure(figsize=(4, 4))
                                ax = plt.gca()
                                ax.plot(behav_pos[beg[i]:end[i]+1, 0]/2-0.5, behav_pos[beg[i]:end[i]+1, 1]/2-0.5, marker='o', markersize=3, markeredgewidth=0, color='k')
                                DrawMazeProfile(axes=ax, color='k', maze_type=trace['maze_type'])
                                ax.invert_yaxis()
                                plt.show()
                                raise ValueError(
                                    f"{mouse}, S{s}, lap {i}:"
                                    f"  k = {k}\n"
                                    f"  {n_inserted} nodes, ie {inserted_nodes}, inserted between {lap_nodes[k]} and {lap_nodes[k+1]}.\n"
                                    f"  Neighbors of {lap_nodes[k]}: {G[lap_nodes[k]]}\n"
                                    f"  Neighbors of {lap_nodes[k+1]}: {G[lap_nodes[k+1]]}\n"
                                )
                        finished_k = k + n_inserted + 1
                        
                        lap_nodes = lap_nodes[:k+1] + inserted_nodes + lap_nodes[k+1:]
                        lap_speed = lap_speed[:k+1] + np.linspace(lap_speed[k], lap_speed[k+1], n_inserted+2)[1:-1].tolist() + lap_speed[k+1:]
                        lap_lap = lap_lap[:k+1] + [lap_lap[k]]*n_inserted + lap_lap[k+1:]
                        lap_t = lap_t[:k+1] + np.linspace(lap_t[k], lap_t[k+1], n_inserted+2)[1:-1].tolist() + lap_t[k+1:]
                        break
                    
                    finished_k = k + 1
        
        if is_discarded_lap:
            continue
        
        # Double Check:
        for k in range(len(lap_nodes)-1):
            if lap_nodes[k+1] not in G[lap_nodes[k]]:
                raise ValueError(
                    f"After insertion, nodes {lap_nodes[k]} and {lap_nodes[k+1]} are still not spatially connected in lap {i}.\n"
                    f"Neighbors of {lap_nodes[k]}: {G[lap_nodes[k]]}\n"
                    f"Neighbors of {lap_nodes[k+1]}: {G[lap_nodes[k+1]]}\n"
                )
        if lap_nodes[-1] != 144:
            raise ValueError(f"Lap {i} does not end with node 144 after processing.")
        if lap_nodes[0] != 1:
            raise ValueError(f"Lap {i} does not start with the correct node after processing.")
        # Assure no consecutive bins are the same.
        assert np.where(np.diff(lap_nodes) == 0)[0].size == 0, f"Lap {i} has consecutive nodes that are the same after processing."
        
        processed_lap.extend(lap_lap)
        processed_nodes.extend(lap_nodes)
        processed_speed.extend(lap_speed)
        
    return (
        np.array(processed_lap, np.int64), 
        np.array(processed_nodes, np.int64), 
        np.array(processed_speed, np.float64)
    )
    
save_dir = r"D:\Softwares\Anaconda2025\envs\models\Lib\site-packages\rtrv_models\data"
for mouse in [10212, 10224, 10227, 10232]:
    print(f"Processing mouse {mouse}...")
    process = {
        "MouseID": [],
        "Session": [],
        "Nodes": [],
        "Speed": [],
        "Lap": [],
        "Maze Type": []
    }
    
    for maze_type in [1, 2]:
        idx = np.where((f1['MiceID'] == mouse)&(np.isin(f1['Stage'], ['Stage 1', 'Stage 2']))&(f1['maze_type'] == maze_type))[0]
        
        if mouse == 10212:
            if maze_type == 1:
                idx = idx[:13] # Mouse 10212 only train 13 sessions of Maze 1 befor being subjected to DSP task.
            else:
                # Maze 2 is not trained before DSP, so skip it.
                continue
        
        assert len(idx) <= 26, f"{idx.shape}"
        for s, file in enumerate(f1['Trace File'][idx]):
            
            if f1['include'][idx[s]] == 0:
                continue
            
            with open(file, 'rb') as f:
                trace = pickle.load(f)
            res = behavioral_info_per_mouse_pretrain(trace, s=s)
            process['MouseID'].append([mouse]*len(res[0]))
            process['Session'].append([s]*len(res[0]))
            process['Nodes'].append(res[1].tolist())
            process['Speed'].append(res[2].tolist())
            process['Lap'].append(res[0].tolist())
            process['Maze Type'].append([maze_type] * len(res[0]))
        
    for key in process:
        process[key] = np.concatenate(process[key])
    
    with open(os.path.join(save_dir, f"Pretrained_{mouse}.pkl"), 'wb') as f:
        pickle.dump(process, f)

d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing mouse 10212...


  0%|          | 0/19 [00:00<?, ?it/s]d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 36/36 [00:00<00:00, 1091.20it/s]


Processing mouse 10224...


  0%|          | 0/37 [00:00<?, ?it/s]d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
  0%|          | 0/41 [00:00<?, ?it/s]d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 37/37 [00:00<00:00, 1156.26it/s]


Processing mouse 10227...


  0%|          | 0/26 [00:00<?, ?it/s]d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 38/38 [00:00<00:00, 622.99it/s]


Processing mouse 10232...


  0%|          | 0/22 [00:00<?, ?it/s]d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
  0%|          | 0/66 [00:00<?, ?it/s]d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\Softwares\Anaconda2025\envs\maze\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 65/65 [00:00<00:00, 1326.42it/s]
